In [18]:
# importando os scripts de consulta

import sys
import os

# Altera o caminho para a pasta raiz do projeto (um nível acima)
raiz_projeto = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Adiciona a raiz do projeto ao caminho de busca do Python se ainda não estiver lá
if raiz_projeto not in sys.path:
    sys.path.append(raiz_projeto)

    
import pandas as pd
from scripts.crm_inscricoes import load_inscricoes
from scripts.crm_leads import load_leads

In [2]:
# ignrando consultores que já saíram da empresa
consultores_ignorar = ['Lorena', 'Breno']

In [17]:
leads = load_leads()



Página 1: 250 registros
Página 2: 250 registros
Página 3: 250 registros
Página 4: 250 registros
Página 5: 250 registros
Página 6: 250 registros
Página 7: 250 registros
Página 8: 250 registros
Página 9: 250 registros
Página 10: 250 registros
Página 11: 250 registros
Página 12: 250 registros
Página 13: 250 registros
Página 14: 250 registros
Página 15: 250 registros
Página 16: 250 registros
Página 17: 250 registros
Página 18: 250 registros
Página 19: 250 registros
Página 20: 250 registros
Página 21: 250 registros
Página 22: 250 registros
Página 23: 250 registros
Página 24: 250 registros
Página 25: 250 registros
Página 26: 250 registros
Página 27: 250 registros
Página 28: 250 registros
Página 29: 250 registros
Página 30: 250 registros
Página 31: 250 registros
Página 32: 250 registros
Página 33: 224 registros


In [19]:
inscricoes = load_inscricoes()

Página 1: 250 registros
Página 2: 250 registros
Página 3: 250 registros
Página 4: 250 registros
Página 5: 250 registros
Página 6: 250 registros
Página 7: 250 registros
Página 8: 250 registros
Página 9: 250 registros
Página 10: 250 registros
Página 11: 250 registros
Página 12: 250 registros
Página 13: 250 registros
Página 14: 250 registros
Página 15: 250 registros
Página 16: 250 registros
Página 17: 212 registros


In [59]:

captacao = leads.merge(inscricoes, how='right', left_on='Inscrição Atual', right_on='Numero Inscricao')

In [62]:
captacao = captacao[
    (captacao['Situação Inscrição'] == 'Matriculado') &
    (captacao['Curso ID'] != "Medicina")]

In [63]:
captacao.shape

(790, 44)

In [35]:

url = 'https://docs.google.com/spreadsheets/d/13DBRdUsHT5wupKkq3Omi9VUr9Nj3R5gYVMfSSkmtXQM/export?format=xlsx'

matriculas_itabirito = pd.read_excel(url, sheet_name = "ITABIRITO")
matriculas_altamira = pd.read_excel(url, sheet_name = "ALTAMIRA")
matriculas_lorena = pd.read_excel(url, sheet_name = "LORENA")
matriculas_canaa = pd.read_excel(url, sheet_name = "Canaã")


matriculas_planilha = pd.concat([matriculas_itabirito, matriculas_altamira, matriculas_lorena, matriculas_canaa], ignore_index=True)

In [36]:
matriculas_planilha.dropna(inplace = True, subset = ['Número de Inscrição'])

In [37]:
matriculas_planilha.shape

(783, 23)

In [64]:
colunas_captacao = ['Nome Completo_x', 'cpf', 'ID Lead', 'Proprietário', 'Numero Inscricao', 'Curso ID', 'Situação Inscrição']
captacao = captacao[colunas_captacao]

In [65]:
captacao.shape

(790, 7)

## Mapeando leads sem proprietário e buscando na planilha

In [66]:
# leads sem proprietário

matriculas_sem_proprietario = captacao[captacao['Proprietário'].isnull() ]

In [67]:
matriculas_sem_proprietario.shape

(39, 7)

In [78]:
df = pd.merge(matriculas_sem_proprietario, matriculas_planilha, how='left', left_on='Numero Inscricao', right_on='Número de Inscrição')

In [82]:
df = df[
    [
    'Nome Completo_x', 
    'cpf',
    'ID Lead', 
    'Proprietário',
    'NOME',
    'CONSULTOR (a)'
    ]
    ].loc[
        ~df['CONSULTOR (a)'].isnull() & ~df['CONSULTOR (a)'].isin(consultores_ignorar)
        ]

In [83]:
df.shape

(29, 6)

In [84]:
df.head()

,Nome Completo_x,cpf,ID Lead,Proprietário,NOME,CONSULTOR (a)
0,Isabela Victoria Ribeiro Cirqueira,711.050.201-50,{5C53F411-F370-F111-AB0F-00224836BDFB},NaN,Isabela Victoria Ribeiro Cirqueira,Gabriel
1,Heitor Silva dos Santos,075.289.292-47,{19A98D0F-0E97-F111-B8DC-6045BD39BC63},NaN,Heitor Silva dos Santos,Geovana
2,Ryllary Silva da Cruz,018.021.572-80,{BD136037-9196-F111-B8DC-00224836B90D},NaN,Ryllary Silva da Cruz,Geovana
3,Thamyris Biancardi Schueroff,025.480.562-01,{7DCCA1AA-8096-F111-B8DC-000D3AC0F764},NaN,Thamyris Biancardi Schueroff,Geovana
4,NaN,NaN,NaN,NaN,Pamela Alves da silva,Daniel


## Mapear matrículas que diverge o proprietário do consultor

In [85]:
df = pd.merge(captacao, matriculas_planilha, how='left', left_on='Numero Inscricao', right_on='Número de Inscrição')

In [87]:


proprietarios_consultores = {
    'Gabriel Matos': 'Gabriel',
    'Maria Eduarda Freire': 'Duda',
    'Maria Rafaela da Rocha Tavares': 'Maria Rafaela',
    'Adauri Marques Delfino Junior': 'Adauri',
    'Karen Flávia': 'Flávia',
    'Daniel Faria Santos': 'Daniel',
    'Geovana Souza da Silva': 'Geovana',
    'Agnata Pamela Silva Souza': 'Agnata',
    'Giovanna Eduarda Do Carmo Braga': 'Giovanna Braga',
    'Hailton Alves Júnior': 'Hailton',
    'Karolaine de Oliveira Alves Souza': 'Karolaine',
    'Thiago Oliveira da Cruz': 'Thiago',
    'Maria Fernanda Luiz Monteiro': 'Maria Fernanda ',
    'Ana Carolina Guimarães': 'Ana Carolina',
    'Lorena Viana Gonçalves Pereira': 'Lorena',
    'Thalia Gabriela de Araújo Silva': 'Thalia',
    'Camila Guimarães Santos': 'Camila',
    'Breno Assis Ferreira': 'Breno'
}

# verificando se o proprietário é diferente do consultor
df['Proprietário'] = df['Proprietário'].map(proprietarios_consultores)

# tirando linhas onde proprietário ou consultor são nulos
df = df.dropna(subset=['Proprietário', 'CONSULTOR (a)'])
divergencias = df[df['Proprietário'] != df['CONSULTOR (a)']][['Nome Completo_x', 'cpf', 'ID Lead', 'Proprietário', 'NOME', 'CONSULTOR (a)']]
display(divergencias.shape)



df = df[~df['CONSULTOR (a)'].isin(consultores_ignorar)]


(0, 6)